# Qwen3-VL-8B — Authority x engagement grid (metrics/realistic)

Both posts make the **same, correct** claim. One is from **"Dr. Remy Ashford" (verified)**, the
other from plain **"Remy Ashford"**. Engagement is varied independently on each side across the
study's seven levels (0, 10, 100, 1K, 10K, 100K, 1M), giving a 7x7 grid.

| region | cells | question |
|---|---|---|
| diagonal (equal engagement) | 7 | does the authority preference survive as popularity rises? |
| conflict (Dr. has fewer) | 21 | how much popularity outweighs a "Dr."? |
| aligned (Dr. has more) | 21 | do the cues saturate, and is the effect symmetric? |

Cells run **diagonal -> conflict -> aligned**, most informative first. The run resumes from a
partial output file, so it can be stopped at any point and still give a complete, analysable
design.

The no-authority control already exists: the off-diagonal cells of
`e1_results_metrics_correct_vs_correct_paired.json` are the same engagement contrasts between two
plain posts. The (0, 0) cell reproduces Stage 1's `pure_authority_both_correct` pairing exactly,
including its A/B slot assignment.


In [ ]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "tokenizers>=0.22.0,<=0.23.0",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

Restart the kernel after running the setup cell above.


In [ ]:
!nvidia-smi

In [ ]:
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Loaded successfully")

In [ ]:
from e1_utils.inference_qwen import run_inference_qwen

In [ ]:
INFERENCE_FN = run_inference_qwen

## Configuration — asserts every stimulus directory before running


In [ ]:
from pathlib import Path
import json

from e1_utils.e1_optimized import LIKE_PROMPT_PAIR
from e1_utils.e1_profile_grid import (run_e1_profile_grid_paired, analyse_profile_grid,
                                      build_grid_cells)

EXPERIMENT_DIR = Path().resolve().parent          # experiments/e1_authority_grid/
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42

# the same 100 posts as the main study and the Stage 1 authority run
selected_numbers = json.loads(
    (EXPERIMENT_DIR / "selected_images.json").read_text())["selected_numbers"]

DR_METRICS     = ROOT_DIR / "benchmarking/correct/dr-remy-ashford/metrics/realistic"
PLAIN_METRICS  = ROOT_DIR / "benchmarking/correct/remy-ashford/metrics/realistic"
DR_BASELINE    = ROOT_DIR / "benchmarking/correct/dr-remy-ashford"       # scale 0
PLAIN_BASELINE = ROOT_DIR / "benchmarking/correct/remy-ashford"          # scale 0

CELLS = build_grid_cells()

# fail here, not 4900 trials in
for d in (DR_METRICS, PLAIN_METRICS, DR_BASELINE, PLAIN_BASELINE):
    assert d.is_dir(), f"missing stimulus dir: {d}"
probe = selected_numbers[0]
for dr_s, pl_s, _ in CELLS:
    for base, bl, s in ((DR_METRICS, DR_BASELINE, dr_s), (PLAIN_METRICS, PLAIN_BASELINE, pl_s)):
        d = bl if s == 0 else base / str(s)
        assert (d / f"{probe}_remy_ashford_c.png").exists(), f"missing stimulus: {d}"

print(f"{len(selected_numbers)} posts x {len(CELLS)} cells "
      f"= {len(selected_numbers) * len(CELLS)} trials")


## Run the grid (diagonal → conflict → aligned; resumable)


In [ ]:
import time
start = time.time()

run_e1_profile_grid_paired(
    selected_numbers,
    dr_metrics_base=DR_METRICS, plain_metrics_base=PLAIN_METRICS,
    dr_baseline_dir=DR_BASELINE, plain_baseline_dir=PLAIN_BASELINE,
    model=model, processor=processor, device=device,
    output_dir=OUTPUT_DIR, seed=SEED, prompt=LIKE_PROMPT_PAIR,
    output_filename="e1_results_authority_grid_metrics.json",
    inference_fn=INFERENCE_FN, cells=CELLS)

elapsed = time.time() - start
print(f"\n⏱ Total runtime: {int(elapsed // 3600)}h "
      f"{int(elapsed % 3600 // 60)}m {int(elapsed % 60)}s")


## Results — read the position split, not just the aggregate


In [ ]:
analyse_profile_grid(OUTPUT_DIR, "e1_results_authority_grid_metrics.json")
